In [1]:
import pyspark 
print(pyspark.__version__)

3.5.0


In [2]:

from pyspark.sql import SparkSession 

spark = (
    SparkSession
    .builder
    .appName("MyfirstRepo")
    .master("spark://spark-master:7077")
    .config("spark.streaming.stopGracefulOnShutdown", True)
    .config("spark.cores.max", "4")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
    )
    .config("spark.sql.shuffle.partitions", 8)
    .getOrCreate()
)
spark

In [3]:
spark.conf.set("fs.s3a.endpoint", "http://minio:9000")
spark.conf.set("fs.s3a.access.key", "minioadmin")
spark.conf.set("fs.s3a.secret.key", "minioadmin")
spark.conf.set("fs.s3a.path.style.access", "true")
spark.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.conf.set("fs.s3a.aws.credentials.provider",
               "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

In [4]:
# df_raw_listen_events = spark.readStream \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", "kafka1:9092,kafka2:9092,kafka3:9092") \
#     .option("subscribe", "listen_events") \
#     .option("startingOffsets", "earliest") \
#     .load()


def get_df_raw(topic: str):
    df_raw = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka1:9092,kafka2:9092,kafka3:9092") \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest") \
        .load()
    return df_raw
    
    

In [5]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

schema_listen_events = StructType([
    StructField("artist",        StringType(),  True),
    StructField("song",          StringType(),  True),
    StructField("duration",      DoubleType(),  True),
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("auth",          StringType(),  True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("userId",        IntegerType(), True),
    StructField("lastName",      StringType(),  True),
    StructField("firstName",     StringType(),  True),
    StructField("gender",        StringType(),  True),
    StructField("registration",  LongType(),    True),
])

schema_status_change_events = StructType([
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("userId",        IntegerType(), True),
    StructField("lastName",      StringType(),  True),
    StructField("firstName",     StringType(),  True),
    StructField("gender",        StringType(),  True),
    StructField("registration",  LongType(),    True),
    StructField("success",       BooleanType(), True),
])

schema_auth_events = StructType([
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("success",       BooleanType(), True),
])

schema_page_view_events = StructType([
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("page",          StringType(),  True),
    StructField("auth",          StringType(),  True),
    StructField("method",        StringType(),  True),
    StructField("status",        IntegerType(), True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("userId",        IntegerType(), True),
    StructField("lastName",      StringType(),  True),
    StructField("firstName",     StringType(),  True),
    StructField("gender",        StringType(),  True),
    StructField("registration",  LongType(),    True),
    StructField("artist",        StringType(),  True),
    StructField("song",          StringType(),  True),
    StructField("duration",      DoubleType(),  True),
])



def get_df_parsed(topic: str, schema: StructType):
    # Read raw data 
    df_raw = get_df_raw(topic)
    df_parsed = df_raw.select(
        from_json(col("value").cast("string"), schema).alias("data")
        ).select("data.*")
    df_parsed.printSchema()
    return df_parsed


# df_parsed = df_raw.select(
#     from_json(col("value").cast("string"), schema).alias("data")
# ).select("data.*")

# df_parsed.printSchema()

In [6]:
import boto3
from botocore.client import Config

def ensure_bucket(bucket_name: str):
    s3 = boto3.client(
        "s3",
        endpoint_url="http://minio:9000",
        aws_access_key_id="minioadmin",
        aws_secret_access_key="minioadmin",
        config=Config(signature_version="s3v4"),
        region_name="us-east-1"
    )
    
    try:
        s3.head_bucket(Bucket=bucket_name)
        print(f"Bucket '{bucket_name}' existed")
    except:
        s3.create_bucket(Bucket=bucket_name)
        print(f"Bucket '{bucket_name}' was built")


# Tạo bucket cho tất cả topics
topics = [
    "listen-events",
    "page-view-events",
    "auth-events",
    "status-change-events",
]

for bucket in topics:
    ensure_bucket(bucket)

# List topics : 
# - auth_events
# - listen_events
# - page_view_events
# - status_change_events


Bucket 'listen-events' existed
Bucket 'page-view-events' existed
Bucket 'auth-events' existed
Bucket 'status-change-events' existed


In [7]:
def get_query(topic: str, schema: StructType):
    # Parse JSON
    df_parsed = get_df_parsed(topic, schema)
    
    # Tên bucket từ tên topic (thay _ bằng -)
    bucket = topic.replace("_", "-")
    
    # Ghi vào MinIO
    query = df_parsed.writeStream \
        .format("parquet") \
        .option("path", f"s3a://{bucket}/data/") \
        .option("checkpointLocation", f"s3a://{bucket}/checkpoints/") \
        .trigger(processingTime="30 seconds") \
        .start()
    
    return query  # không awaitTermination ở đây


# Chạy tất cả topics
queries = [
    get_query("listen_events",        schema_listen_events),
    get_query("page_view_events",     schema_page_view_events),
    get_query("auth_events",          schema_auth_events),
    get_query("status_change_events", schema_status_change_events),
]

# Chờ tất cả cùng lúc
spark.streams.awaitAnyTermination()

root
 |-- artist: string (nullable = true)
 |-- song: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- ts: long (nullable = true)
 |-- sessionId: integer (nullable = true)
 |-- auth: string (nullable = true)
 |-- level: string (nullable = true)
 |-- itemInSession: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- state: string (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- userId: integer (nullable = true)
 |-- lastName: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- registration: long (nullable = true)

root
 |-- ts: long (nullable = true)
 |-- sessionId: integer (nullable = true)
 |-- page: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- level: string (nullable = true)
 |-

StreamingQueryException: [STREAM_FAILED] Query [id = a2f5d83e-1e5f-4598-8b53-7e1eab920c27, runId = b35eaa0a-1448-4083-82c2-c815302e1ff0] terminated with exception: [BATCH_METADATA_NOT_FOUND] Unable to find batch s3a://auth-events/data/_spark_metadata/0.